# Interactive Network

## Graph Display Setup

In [1]:
# Imports

#Network stuff
import networkx as nx
from pyvis.network import Network
from matplotlib.colors import to_hex

#Widgets and display stuff
import ipywidgets as widgets
from IPython.display import HTML, clear_output
import pdfkit

#basics
import sys
import os

import pandas as pd
import numpy as np

import itertools

#Import utils functions
sys.path.append(os.path.abspath("../"))
import utils

/usr3/graduate/rowanon/.local/lib/python3.10/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr3/graduate/rowanon/.local/lib/python3.10/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr3/graduate/rowanon/.local/lib/python3.10/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr3/graduate/rowanon/.local/lib/python3.10/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


### Load datasets

In [2]:
# Experimental
dataset_experimental = pd.read_csv("./datasets/interaction-terms_experimental.csv", index_col=0).apply(pd.to_numeric)

# CRM Informed
## zscored
### Normed by growth
dataset_crmgznorm = pd.read_csv("./datasets/interaction-terms_CRMgznorm.csv", index_col=0).apply(pd.to_numeric)

## non-zscored
### Normed by growth
dataset_crmgnorm = pd.read_csv("./datasets/interaction-terms_CRMgnorm.csv", index_col=0).apply(pd.to_numeric)
### Non-normed
dataset_crm = pd.read_csv("./datasets/interaction-terms_CRM.csv", index_col=0).apply(pd.to_numeric)

datasets = {
    'Experimental': dataset_experimental,
    'CRM informed, zscore and g normalized': dataset_crmgznorm,
    'CRM informed, g normalized': dataset_crmgnorm,
    'CRM informed': dataset_crm
}

### Network functions

#### Construct whole network

In [3]:
def construct_whole_network(df, threshold=None):
    """
    Creates a whole network with all nodes and edges between species.
    
    input: dataframe where
    output: Network x graph, G
    """
    G = nx.DiGraph()
    
    # Add nodes (all species)
    for item in df.index:
        G.add_node(item, group="metabolite")
        
    # Add nodes (all metabolites)
    for item in df.columns.tolist():
        if item in utils.sps_to_name.keys():
            name = utils.sps_to_name[str(item)]
        else:
            name = item
        G.add_node(name, group="species")
    
    #Add edges between species and metabolite nodes
    
    df.reset_index(inplace=True)
    
    df.rename(columns={df.columns[0]: "Metabolite"}, inplace=True)
    
    df_long = pd.melt(df, id_vars=["Metabolite"], var_name="Species", value_name="Interaction")
    
    
    # Set thresholds for interaction terms
    if threshold:
        df_filtered = df_long[abs(df_long["Interaction"]) >= threshold].copy()
    else:
        df_filtered = df_long
    
    for idx, row in df_filtered.iterrows():
        species_code = row["Species"]
        species_label = utils.sps_to_name.get(str(species_code), str(species_code))
        metabolite = row["Metabolite"]
        weight = row["Interaction"]
        if weight > 0:
            G.add_edge(species_label, metabolite, weight=abs(weight), group='positive')
        elif weight < 0:
            G.add_edge(metabolite, species_label, weight=abs(weight), group='negative')
    
    return G

#### Trim to selected species, n interactions

In [4]:
def trim_network_species(G, sps_list):
    """ Keep a subgraph of just the selected species (and metabolite nodes) """

    keep = []

    for node in G.nodes:
        if G.nodes[node]['group'] == 'metabolite':
            keep.append(node)
        elif node in sps_list:
            keep.append(node)
        else:
            continue
    
    G = G.subgraph(keep).copy()
              
    return G

def trim_network_topn(G, n, pos_neg = False):
    """Trim by weight values, keeping top n edges for final network."""
    if pos_neg:
        #top positive edges
        edge_values_pos = [data['weight'] for u, v, data in G.edges(data=True) if 'weight' in data and data['group'] == 'positive']
        edge_values_pos.sort(reverse=True)

        threshold = min(edge_values_pos[:n])

        edges_to_keep_pos = [(u, v, data) for u, v, data in G.edges(data=True) if data['weight'] > threshold and data['group'] == 'positive']

        #top negative edges

        edge_values_neg = [data['weight'] for u, v, data in G.edges(data=True) if 'weight' in data and data['group'] == 'negative']
        edge_values_neg.sort(reverse=True)

        threshold = min(edge_values_neg[:n])

        edges_to_keep_neg = [(u, v, data) for u, v, data in G.edges(data=True) if data['weight'] > threshold and data['group'] == 'negative']
        
        edges_to_keep = edges_to_keep_neg + edges_to_keep_pos
        
    else:
        edge_values = [data['weight'] for u, v, data in G.edges(data=True) if 'weight' in data]
        edge_values.sort(reverse=True)
        
        threshold = min(edge_values[:n])

        edges_to_keep = [(u, v, data) for u, v, data in G.edges(data=True) if data['weight'] > threshold]
    
    #Create empty network (no edges) and add the ones we curated
    G = nx.create_empty_copy(G).copy()

    for u, v, data in edges_to_keep:
        G.add_edge(u, v, **data)
    
    return G
    
def trim_network_clean(G, remove_null, remove_empty, dead_end):
    """Clean up, removing some nodes or edges based on user specification"""
    
    # Remove edges that are being produced by all species in the set.
    if remove_null:
        edges_to_remove = []
        for node, data in G.nodes(data=True):
            if data.get('group') == 'metabolite':
                in_edges = G.in_edges(node)

                if len(in_edges) == G.degree(node): #If all the edges are in edges, remove them.
                    in_edges = [(u,v) for u, v in in_edges]
                    edges_to_remove = edges_to_remove + in_edges
    
        G.remove_edges_from(edges_to_remove)
        
    if dead_end:
        dead_end = []
        for node, data in G.nodes(data=True):
            if data.get('group') == 'metabolite':
                if G.degree(node) == 1:
                    dead_end.append(node)
                
        G.remove_nodes_from(dead_end)
    
    # Identify isolated nodes
    if remove_empty:
        empty_nodes = [node for node, degree in G.degree() if degree < 1]

        # Remove the isolated nodes
        G.remove_nodes_from(empty_nodes)
        
    
    return G
    

#### Custom layouts

In [5]:
def pairwise_long_layout(G):
    # Place metabolite nodes in the middle.
    posx = []
    posy = []
    n_met = 1
    n_spe = 1
    n_nodes = 0

    for node in G.nodes:
        n_nodes += 1
        if G.nodes[node]['group'] == 'metabolite':
            posx.append(2)
            posy.append(n_met)
            n_met += 1
        elif G.nodes[node]['group'] == 'species':
            if n_spe == 1:
                posx.append(1)
            elif n_spe == 2:
                posx.append(3)
            else:
                print("Too many species in the network provided.")
            posy.append(1000)
            n_spe += 1

    #Replace large placeholder with 
    posy = [n_met / 2 if x == 1000 else x for x in posy]

    pos = {node:[posx[i], posy[i]] for i, node in enumerate(G.nodes)}
    
    return pos

#### Construct pyvis network from networkx graph

In [6]:
def pyvis_from_nx(G, gravity, spring, no_species_label, no_metabolite_label):
    
    # Something modifies G in place in here... not sure what.
    net = Network(notebook=True, height="750px", width="100%", bgcolor="white", font_color="black", directed=True)
    
    # Graph attributes
    #net.barnes_hut(gravity=gravity, spring_length=spring)
    
    net.from_nx(G)
    
    for edge in G.edges(data=True):
        source, target, data = edge
        weight = data['width'] # Get weight, default to 'N/A' if not found
        
        # Get pyvis edge and populate with info from original graph
        for pv_edge in net.edges:
            if pv_edge['from'] == source and pv_edge['to'] == target:
                pv_edge['title'] = f"Weight: {weight}"
                pv_edge['value'] = abs(int(weight))
                
                if edge[2]['group'] == 'negative':
                    pv_edge['color'] = 'blue'
                    
                elif edge[2]['group'] == 'positive':
                    pv_edge['color'] = 'red'
                    
                break
                
    # Modify node style
    # Get color dict
    # Species color map
    color_dict = utils.get_species_colormap()
    
    met_group = utils.load_met_classes()
    
    #Metabolite color map
    color_map = {"Sugar":"pink", "Organic_Acid":"goldenrod", "Amino_Acid":"mediumseagreen",
                 "Nucleobase":"darkturquoise", "Others":"orchid"}
    rev = { v: k for k, vals in met_group.items() for v in vals }
    
    for node in net.nodes:
        name = node['id']
        if node['group'] == 'species':
            # Color by species
            node['shape'] = 'dot'
            node['color'] = to_hex(color_dict[name])
            if no_species_label:
                node['label'] = ""
            
        elif node['group'] == 'metabolite':
            # Color by metabolite type            
            node['shape'] = 'triangle'
            node['color'] = color_map[rev[name]]
            if no_metabolite_label:
                node['label'] = ""
        
    return net

## Run Interactive Display

In [7]:

#### WIDGETS ##################################################################

gravity = widgets.FloatSlider(min=0, max=5, step=0.1, value=0.2,description="Gravity")
spring  = widgets.IntSlider(min=0, max=300, step=10, value=100,description="Spring length")

generate = widgets.Button(description="Generate")

save_html = widgets.Button(description="Save HTML")
save_pdf = widgets.Button(description="Save PDF")

file_name = widgets.Text(value='default', placeholder='Type something', description='Filename:', disabled=False)

sepn = widgets.Checkbox(value=False, description='Use top n positive and negative interactions?', disabled=False, indent=False)
empty = widgets.Checkbox(value=True, description='Remove empty nodes?', disabled=False, indent=False)
null = widgets.Checkbox(value=False, description='Remove nodes with null interactions (all edges are production)?', disabled=False, indent=False)
dead_end = widgets.Checkbox(value=False, description='Remove nodes that are dead-ends?', disabled=False, indent=False)


metlabel = widgets.Checkbox(value=False, description='Hide metabolite labels?', disabled=False, indent=False)
spslabel = widgets.Checkbox(value=False, description='Hide species labels?', disabled=False, indent=False)


n = widgets.IntText(value=20, description='n top edges to keep:', disabled=False, layout={'width': '150px'})

species = widgets.SelectMultiple(
            options=utils.sps_names,
            value=['Lysobacter'],
            rows=15,
            description='Species',
            disabled=False
        )

dataset = widgets.Select(
            options=list(datasets.keys()),
            value=list(datasets.keys())[0],
            discription="Dataset:",
            disabled=False,
            layout={'width': '300px'}
        )

out = widgets.Output()

#### MODULE VARS ##############################################################

last_net = None

#### ON CLICK (BUTTON) DEFINITIONS ############################################

def save_html_onclick(b):
    if last_net is None:
        with out:
            print("Please generate a network first!")
        return
    else:
        last_net.write_html(f"{file_name.value}.html")
    
def save_pdf_onclick(b):
    if last_net is None:
        with out:
            print("Please generate a network first!")
        return
    else:
        try:
            pdfkit.from_file("./tmp_graph.html", f"{file_name.value}.pdf")
        except Exception as e:
            with out:
                print("Please load module `wkhtmltopdf` in order to save as pdf.")

########### MAIN ##############################################################
def generate_onclick(b):
    global last_net
    with out:
        clear_output()
        
        # Call functions to create network based on values
        
        # Handle dataset:
        df = datasets[dataset.value].copy()
        
        # Construct whole network from dataset:
        G = construct_whole_network(df)
        
        # Selected species =
        selected_sps = list(species.value)
        
        if len(selected_sps) != 15:
            #Trim network based on selected species
            G = trim_network_species(G, selected_sps)
        
        G = trim_network_topn(G, n.value, sepn.value)
        
        # Clean up
        G = trim_network_clean(G, null.value, empty.value, dead_end.value)
        
        # Construct pyvis from networkx
        net = pyvis_from_nx(G, gravity.value, spring.value, spslabel.value, metlabel.value)
        
        # write to a temp file and display
        tmp = "tmp_graph.html"
        net.write_html(tmp)
        display(HTML(open(tmp).read()))
        
        last_net = net

###############################################################################        
        
        
generate.on_click(generate_onclick)

#Saving funcs
save_html.on_click(save_html_onclick)
save_pdf.on_click(save_pdf_onclick)


widgets.VBox(
    children = [
        widgets.HBox([gravity, spring]), 
        widgets.HBox(
            children = [widgets.HBox([species, dataset]),
                       widgets.VBox([sepn, empty, null, metlabel, spslabel, dead_end, n])]
            ),
        widgets.VBox([generate, out]),
        widgets.HBox([file_name, save_html, save_pdf])
        ]
    )

